In [4]:
import os
import pandas as pd
import numpy as np
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE,RandomOverSampler
import torch
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
import random
from pathlib import Path

In [ ]:
def calculate_dtw_matrix(file_path, window_size=60, step_size=30):
    data = pd.read_csv(file_path,header=None)
    voltage_data = data.iloc[:, 7].values
    other_data = data.iloc[:, :7].values
    dtw_matrix = []
    for i in range(0, len(voltage_data) - window_size, step_size):
        voltage_window = voltage_data[i:i + window_size]
        other_window = other_data[i:i + window_size, :]
        dtw_distances = []
        for j in range(7):
            dist, _ = fastdtw(voltage_window.reshape(-1, 1), other_window[:, j].reshape(-1, 1), dist=euclidean)
            dtw_distances.append(dist)
        dtw_matrix.append(dtw_distances)
    return np.array(dtw_matrix)

def create_dataset(file_paths, labels, window_size=60, step_size=30):
    data = []
    label = []

    for i, file_path in enumerate(file_paths):
        dtw_matrix = calculate_dtw_matrix(file_path, window_size, step_size)
        data.append(dtw_matrix)
        label.append(labels[i])

    X = np.stack(data, axis=0)
    y = np.array(label)

    return X, y

# Label map
label_map = {
    'short_5.csv': 3, 'short_10.csv': 3, 'short_20.csv': 3, 'short_30.csv': 3,
    'short_50.csv': 3, 'short_75.csv': 3, 'short_85.csv': 3, 'short_100.csv': 2,
    'short_220.csv': 2, 'short_300.csv': 2, 'short_500.csv': 1, 'short_560.csv': 1,
    'short_680.csv': 1, 'short_820.csv': 1, 'short_normal.csv': 0, 'short_normal_1.csv': 0,
    'short_normal_2.csv': 0, 'short_normal_3.csv': 0
}

# Set filepath and labels
file_paths = [f"short_{i}.csv" for i in [5, 10, 20, 30, 50, 75, 85, 100, 220, 300, 500, 560, 680, 820, 'normal', 'normal_1', 'normal_2', 'normal_3']]
labels = [label_map[f"short_{i}.csv"] for i in [5, 10, 20, 30, 50, 75, 85, 100, 220, 300, 500, 560, 680, 820, 'normal', 'normal_1', 'normal_2', 'normal_3']]

# Create dataset
X, y = create_dataset(file_paths, labels)


In [23]:

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4,stratify=y,random_state=42)

# SMOTE
def windowwise_interpolation_resample(X, y, sampling_strategy, random_state=42):
    rng = np.random.default_rng(random_state)

    X_res = [x.copy() for x in X]
    y_res = [label for label in y]

    n_samples, seq_len, input_size = X.shape

    for cls, target_num in sampling_strategy.items():
        cls_indices = np.where(y == cls)[0]
        current_num = len(cls_indices)

        if current_num == 0:
            continue

        need_num = target_num - current_num

        if need_num <= 0:
            continue

        for _ in range(need_num):
            new_seq = np.zeros((seq_len, input_size))

            for t in range(seq_len):
                if current_num == 1:
                    new_seq[t] = X[cls_indices[0], t, :].copy()
                else:
                    i, j = rng.choice(cls_indices, size=2, replace=False)
                    alpha = rng.random()

                    new_seq[t] = X[i, t, :] + alpha * (X[j, t, :] - X[i, t, :])

            X_res.append(new_seq)
            y_res.append(cls)

    return np.array(X_res), np.array(y_res)

#Normalization
scaler = MinMaxScaler()

n_train, seq_len, input_size = X_train.shape
n_test = X_test.shape[0]

X_train_2d = X_train.reshape(-1, input_size)
X_test_2d = X_test.reshape(-1, input_size)

X_train_scaled = scaler.fit_transform(X_train_2d).reshape(n_train, seq_len, input_size)
X_test_scaled = scaler.transform(X_test_2d).reshape(n_test, seq_len, input_size)

#Expanded Dataset
X_train_res, y_train_res = windowwise_interpolation_resample(
    X_train_scaled,
    y_train,
    sampling_strategy={0: 1000, 1: 1000, 2: 1000, 3: 1000},
    random_state=42
)

X_test_res, y_test_res = windowwise_interpolation_resample(
    X_test_scaled,
    y_test,
    sampling_strategy={0: 200, 1: 200, 2: 300, 3: 300},
    random_state=42
)

# Transform to pytorch tensor
X_train_res = torch.tensor(X_train_res, dtype=torch.float32)
y_train_res = torch.tensor(y_train_res, dtype=torch.long)
X_test_res = torch.tensor(X_test_res, dtype=torch.float32)
y_test_res = torch.tensor(y_test_res, dtype=torch.long)

# Create DataLoader
train_dataset = torch.utils.data.TensorDataset(X_train_res, y_train_res)
test_dataset = torch.utils.data.TensorDataset(X_test_res, y_test_res)

def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(7)

train_generator = torch.Generator()
train_generator.manual_seed(7)

test_generator = torch.Generator()
test_generator.manual_seed(8)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    generator=train_generator,
    num_workers=0
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    generator=test_generator,
    num_workers=0
)


In [ ]:
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# GRU model
class GRUModel(nn.Module):
    def __init__(self, input_size=7, hidden_size=128, num_classes=4):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        gru_out, _ = self.gru(x)
        
        if len(gru_out.shape) == 2:
            x = gru_out
        else:
            x = gru_out[:, -1, :]
        
        x = self.flatten(x)
        x = self.fc(x)
        return x  

# Initialize the Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = GRUModel(input_size=7, hidden_size=64, num_classes=4).to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()  # 
optimizer = optim.Adam(model.parameters(), lr=0.003)

# Train proces
num_epochs = 50
train_losses = []
valid_losses = []
train_accuracies = []
valid_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        #Output Class
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    # print the result of each cycle
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%")



In [ ]:
# Output Confusion Matrix
model.eval()
all_labels = []
all_predictions = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

cm = confusion_matrix(all_labels, all_predictions)

# Plot Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Light Short', 'Medium Short', 'Severe Short'], yticklabels=['Normal', 'Light Short', 'Medium Short', 'Severe Short'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()
